# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset package using the `mlcroissant` library. We will load the Croissant schema, inspect its record sets and fields, extract the data for analysis, perform exploratory data steps, and visualize key properties.

### Dataset Source

The dataset source is provided via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install the mlcroissant package if not already installed
!pip install mlcroissant

## 1. Data Loading

Let's load the dataset using `mlcroissant`, inspect its metadata, and get initial information about the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Show metadata: name and description
md = dataset.metadata
print(f"{md.name}: {md.description}\n")

# (Optional) Print identifiers
print(f"Identifier: {getattr(md, 'identifier', None)}")
print(f"Version: {getattr(md, 'version', None)}")


## 2. Data Overview

Explore the record sets defined in the Croissant schema. For each record set, we display its `@id`, name, and available fields with their `@id`s. All entities are referenced by their `@id` as required.

In [ ]:
# List all record sets and display their IDs and fields

print('Available record sets:')
record_sets = list(dataset.metadata.record_sets)
record_set_ids = []

for rs in record_sets:
    print(f"  RecordSet name: {rs.name}; @id: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for f in rs.fields:
            print(f"      {f.name} (@id: {f.id}) | dataType: {getattr(f, 'data_type', 'unknown')}")
    else:
        print("    No fields found.")
    print()

# Show the full list of record set @id's
print('Record set @id list:', record_set_ids)

## 3. Data Extraction

We now load the records from each record set as a pandas DataFrame. Replace `<record_set_id>` and `<field_id>` with the real `@id`'s obtained from the output above as needed.

_All uses of entities are referenced by their `@id`._

In [ ]:
# Extract all record sets (@id-based)

dataframes = {}
for rs_id in record_set_ids:
    # Use the @id to retrieve records
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set: {rs_id} | {df.shape[0]} records; Columns: {list(df.columns)}.\n")
    else:
        print(f"Record set {rs_id} has no records.\n")

# Choose main tabular record set (the one with most clinical data)
if len(dataframes) == 0:
    raise Exception('No record sets with data found for extraction.')
main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
print(f"Default to main record set: {main_record_set_id}\n")

# Show the first few rows and the @id's for fields/columns
print('Columns (use these @id strings for referencing fields):')
print(dataframes[main_record_set_id].columns.tolist())
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing and basic exploration steps. We select numeric and categorical fields _using their `@id`_, filter, normalize, and group records using these.

In [ ]:
# Choose a numeric field and a group field by @id from previous output
# These should match the column names from the DataFrame above, which are the actual field @id in the schema.

# For example purposes, assume the following @ids (replace as needed):
numeric_field_id = None
group_field_id = None

df = dataframes[main_record_set_id]

# Guess a numeric field (first field with numeric dtype and 'age' or 'interval')
for c in df.columns:
    if df[c].dtype in [np.int64, np.float64, np.int32, np.float32]:
        numeric_field_id = c
        break
# If none found, try a field containing 'age' or 'interval'
if not numeric_field_id:
    for c in df.columns:
        if any(k in c.lower() for k in ['age', 'interval', 'duration']):
            try:
                # Try converting to numeric
                df[c] = pd.to_numeric(df[c], errors='coerce')
                if df[c].notnull().sum() > 0:
                    numeric_field_id = c
                    break
            except Exception:
                continue

if numeric_field_id is None:
    print('Could not auto-detect a numeric field.')
else:
    print(f'Analyzing numeric field: {numeric_field_id}')

# Guess a group field (e.g., sex, anatomical location, etc.)
potential_group_fields = [c for c in df.columns if c != numeric_field_id and (('sex' in c.lower()) or ('site' in c.lower()) or ('location' in c.lower()) or (df[c].dtype == 'object'))]
for g in potential_group_fields:
    # Pick field with few unique values
    if df[g].nunique() < 10:
        group_field_id = g
        break
if group_field_id is None and len(potential_group_fields) > 0:
    group_field_id = potential_group_fields[0]

if group_field_id is not None:
    print(f'Grouping by field: {group_field_id}')

# Filtering records where numeric_field > threshold
if numeric_field_id:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'O' else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={filtered_df.shape[0]}):\n")
    display(filtered_df.head())

    # Normalization
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Grouped mean by group field (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print('No numeric field found for filtering and EDA.')

## 5. Visualization

Plot the distribution of the selected numeric field and compare across groupings where applicable.

In [ ]:
# Basic visualization of the selected numeric field and (if available) by group
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

This notebook demonstrated how to reference, load, and analyze a Croissant dataset using the `mlcroissant` library. 
- All references to record sets, fields, and columns used their `@id`s for full traceability.
- We loaded the main tabular record set and explored the distribution of a key numeric variable, applying basic filtering, normalization, and visualization.

_For deeper insights, refer to the rich metadata fields and extend the analyses as needed to answer domain-specific clinical or molecular questions related to second primary colorectal cancer._